# NS10657 — F2: bounded preprocessing check

Ulrich (2026-09-29, point 2). Compare the current analysis with (B) a modestly relaxed
counts-ratio cutoff and (C) no ratio cutoff, keeping the independently justified quality
filters unchanged. Report the fitted coefficients and the residual scatter sigma_n per variant,
and the effect of excluding long counter-sensor separations.

Environmental matching uses a GLOBAL time axis (not day by day), so the |dt| statistics agree
with p2_checks C3 (median 0.41 s, p95 4.3 s, max 458 s, ~1.5% > 10 s).

In [12]:
import os, json, glob
from pathlib import Path
import numpy as np, pandas as pd

here = Path.cwd()
root = next(p for p in [here, *here.parents] if (p/'data'/'processed'/'full_data.csv').exists())
RAW  = Path(os.environ.get('NS10657_RAW', '/Volumes/PortableSSD/Freiburg/Actomics/barium_network/zenodo/data/raw/'))
OUT  = here/'NS10657_preprocessing_bound.json'

VARIANTS = {'A_current': (2.2584867, 3.0), 'B_relaxed': (2.2584860, 3.0), 'C_no_cutoff': (None, None)}
TOL_LONG = 10.0
F_FRB, F_1762 = 384.2281145e12, 170.126432e12
print('root', root); print('raw', RAW, RAW.exists())

root /Users/weiwu/Documents/科研相关/4_弗赖堡大学/TIAMO_personal/scripts/DataEvaluation/2025_07_ACTOMICS/zenodo/refractive_index_1762nm_v2.0
raw /Volumes/PortableSSD/Freiburg/Actomics/barium_network/zenodo/data/raw True


In [13]:
cr_files = sorted(glob.glob(str(RAW/'interferometer/counts_ratio_data_*.csv')))
c = pd.concat([pd.read_csv(f) for f in cr_files], ignore_index=True)
c['time'] = pd.to_datetime(c['time'], utc=True, errors='coerce')
c = c.dropna(subset=['time', 'counts_ratio'])
c = c[c['counts_ratio'] > 0].sort_values('time').reset_index(drop=True)
print('counter files', len(cr_files), '| counter rows (all)', len(c))

def env_frame(sensor):
    parts = []
    for f in sorted(glob.glob(str(RAW/f'environmental/{sensor}_data_*.csv'))):
        e = pd.read_csv(f)
        e.columns = ['time', sensor]
        e['time'] = pd.to_datetime(e['time'], utc=True, errors='coerce')
        parts.append(e[['time', sensor]].dropna())
    return pd.concat(parts, ignore_index=True).rename(columns={'time': 't_env'}).sort_values('t_env').reset_index(drop=True)

for sensor in ['temperature', 'humidity', 'pressure']:
    e = env_frame(sensor)
    b = pd.merge_asof(c[['time']], e, left_on='time', right_on='t_env', direction='backward')
    f = pd.merge_asof(c[['time']], e, left_on='time', right_on='t_env', direction='forward')
    dt_b = (c['time'] - b['t_env']).dt.total_seconds().abs()
    dt_f = (f['t_env'] - c['time']).dt.total_seconds().abs()
    take = dt_b <= dt_f
    c[sensor] = np.where(take, b[sensor], f[sensor])
    c['dt_' + sensor] = np.where(take, dt_b, dt_f)
    print(sensor, 'env rows', len(e)); del e, b, f
c['dt_max'] = c[['dt_temperature', 'dt_humidity', 'dt_pressure']].max(axis=1)
c = c.dropna(subset=['temperature', 'humidity', 'pressure']).reset_index(drop=True)

counter files 93 | counter rows (all) 156938
temperature env rows 42885129
humidity env rows 42659828
pressure env rows 42649752


In [15]:
import sys; sys.path.append('..')
from models.nist.refractive_index import ciddor

lam780 = 299792458.0 / F_FRB * 1e9

def _n_1762(row):
    # ciddor() is scalar-only (svp_water uses 'if t >= 0'), so evaluate row by row,
    # as in code/data_processing/preprocess_data.ipynb.
    n_rb = ciddor(wave=lam780, t=row['temperature'], p=row['pressure'] * 100.0, rh=row['humidity'])
    return n_rb * F_FRB / F_1762 / row['counts_ratio']

c['n_1762'] = c.apply(_n_1762, axis=1)
print('n derived for', len(c), 'rows')

n derived for 156938 rows


In [16]:
def fit(df):
    X = np.column_stack([np.ones(len(df)), df['temperature'], df['humidity'], df['pressure']])
    y = df['n_1762'].to_numpy()
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return beta, float((y - X @ beta).std(ddof=4))

def sub(df, lo, hi):
    return df if lo is None else df[(df['counts_ratio'] >= lo) & (df['counts_ratio'] <= hi)]

res = {}
for name, (lo, hi) in VARIANTS.items():
    d = sub(c, lo, hi); beta, s = fit(d)
    res[name] = dict(ratio_window=None if lo is None else [lo, hi], n_rows=int(len(d)), n0=float(beta[0]),
                     alpha_T=float(beta[1]), alpha_H=float(beta[2]), alpha_P=float(beta[3]), sigma_n=s)
    print(name, 'n', len(d), 'sigma_n %.4e' % s)

A_current n 145789 sigma_n 1.8368e-07
B_relaxed n 155145 sigma_n 2.1398e-07
C_no_cutoff n 156938 sigma_n 2.2432e-07


In [17]:
dt = c['dt_max'].to_numpy()
print('|dt| median %.3f s, p95 %.3f s, max %.1f s' % (np.median(dt), np.percentile(dt, 95), dt.max()))
for tol in (10.0, 60.0):
    m = dt > tol; print('|dt|>%.0fs: n_long=%d (%.2f%%)' % (tol, m.sum(), 100*m.mean()))

sub_ok = c[c['dt_max'] <= TOL_LONG]
excl = {}
for name, (lo, hi) in VARIANTS.items():
    b0, s0 = fit(sub(c, lo, hi)); b1, s1 = fit(sub(sub_ok, lo, hi))
    excl[name] = dict(n_rows=int(len(sub(sub_ok, lo, hi))), alpha_H=float(b1[2]), sigma_n=s1,
                      d_alpha_H_pct=float(100*(b1[2]-b0[2])/b0[2]), d_sigma_n_pct=float(100*(s1-s0)/s0))
    print(name, 'excl >%.0fs' % TOL_LONG, excl[name])

|dt| median 0.623 s, p95 5.060 s, max 112.8 s
|dt|>10s: n_long=2917 (1.86%)
|dt|>60s: n_long=10 (0.01%)
A_current excl >10s {'n_rows': 142943, 'alpha_H': -1.3135709932338362e-08, 'sigma_n': 1.836808827016703e-07, 'd_alpha_H_pct': -0.14965644177188558, 'd_sigma_n_pct': 0.0010136511538053434}
B_relaxed excl >10s {'n_rows': 152235, 'alpha_H': -1.3675973737678635e-08, 'sigma_n': 2.1427115991124823e-07, 'd_alpha_H_pct': 0.1455696931170626, 'd_sigma_n_pct': 0.13395550768840356}
C_no_cutoff excl >10s {'n_rows': 154021, 'alpha_H': -1.423295395114469e-08, 'sigma_n': 2.247356531649708e-07, 'd_alpha_H_pct': 0.2973153034343444, 'd_sigma_n_pct': 0.18434773886750574}


In [18]:
result = dict(artifact='NS10657_preprocessing_bound_F2', variants=res, exclusion_refit=excl)
json.dump(result, open(OUT, 'w'), indent=2)
print('wrote', OUT)

wrote /Users/weiwu/Documents/科研相关/4_弗赖堡大学/TIAMO_personal/scripts/DataEvaluation/2025_07_ACTOMICS/zenodo/refractive_index_1762nm_v2.0/code/analysis/NS10657_preprocessing_bound.json


## Notes

- The only difference between variants is the counts-ratio window; the ratio filter is the
  quantity under test, all other filters are unchanged.
- The change induced by relaxing the window appears in BOTH the fitted response (the humidity
  coefficient becomes more negative) and the residual scatter (which rises); the check does not
  by itself establish which of the two is the cause.
- Side observation: relaxing the window moves the humidity coefficient towards the Mathar value
  (the difference from Mathar falls from 24.8% to 21.8% / 18.8%).